In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import SystemMessage

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

In [5]:
#Create LTM store + seed mempries (done BEFORE runnign the graph)
store = InMemoryStore()
user_id = "u1"

#Store user details as as single blog (simple for teaching)
# you can also split into multiple records: this keeps it easy

user_details = ("usser", user_id, "details")

store.put(user_details, "profile_1", {"data": "Name: Milan Rai"})
store.put(user_details, "profile_2", {"data": "Profession: Teaches AI on Youtube"})
store.put(user_details, "profile_1", {"data":"Prefers concise answers"})
store.put(user_details, "profile_2", {"data":"Likes example in python"})
store.put(user_details, "profile_1", {"data":"Building MCp servers (Python based program)"})

In [6]:
# 2. system prompt template (your prompt) 
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize 
your responses based on what you know about the user.

Your goal is to provide relevant, friendly, and tailored 
assistance that reflects the user’s preferences, context, and past interactions.

If the user’s name or relevant personal context is available, always personalize your responses by:
    – Always Address the user by name (e.g., "Sure, Nitish...") when appropriate
    – Referencing known projects, tools, or preferences (e.g., "your MCP  server python based project")
    – Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible. For example, instead of "In TypeScript apps..." 
say "Since your project is built with TypeScript..."

Use personalization especially in:
    – Greetings and transitions
    – Help or guidance tailored to tools and frameworks the user uses
    – Follow-up messages that continue from past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user’s memory (which may be empty) is provided as: {user_details_content}
"""